# Covariate adjustment

Post-survey effects of the condition with and without adjustment for four pre-survey covariates (Section 5, User Perception).


## 0. Setup


In [1]:
from pathlib import Path

import zlib

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 210)

BASE_DIR = (Path(".").resolve() / "../data").resolve()
DB_DIR, FINAL_DIR = BASE_DIR / "database", BASE_DIR / "final"

REACTIVE, PROACTIVE = "Assistant", "Interview"
ARMS = [REACTIVE, PROACTIVE]
N_BOOT = 10_000
SEED = 20260903


def read(path, **kw):
    return pd.read_csv(path, dtype=str, keep_default_na=False, **kw)


def num(s):
    return pd.to_numeric(s.replace("", np.nan), errors="coerce")


demographics = read(FINAL_DIR / "participant_demographics.csv")
codebook = read(FINAL_DIR / "question_labels.csv").set_index("column")
complete = demographics[demographics["status"] == "COMPLETE"].reset_index(drop=True)
assert len(complete) == 49 and complete["condition"].value_counts().to_dict() == {PROACTIVE: 26, REACTIVE: 23}

X = pd.DataFrame({c: num(complete[c]) for c in complete.columns
                  if c.startswith("QID") and not c.endswith("_TEXT")})
cond = complete["condition"]
reactive = (cond == REACTIVE).astype(float).to_numpy()
print(f"{len(complete)} COMPLETE participants: {int(reactive.sum())} reactive, {int((1 - reactive).sum())} proactive")

49 COMPLETE participants: 23 reactive, 26 proactive


## 1. Covariates and outcomes


In [2]:
def reverse(frame, low, high):
    return (low + high) - frame


covariates = pd.DataFrame({
    "female": (complete["QID7"] == "Weiblich").astype(float).to_numpy(),
    "university": (complete["QID13"] == "Universität oder ETH").astype(float).to_numpy(),
    "left-right (0-9)": X["QID23"].to_numpy(),
    "CAVAA attitude, pre (-2..2)": X[["QID32_6", "QID32_7", "QID32_8", "QID32_9"]].mean(axis=1).to_numpy(),
})
assert covariates.notna().all().all(), "a covariate is missing for some participant"

CERTAINTY = {"überhaupt nicht sicher": 1, "nicht so sicher": 2, "sicher": 3, "sehr sicher": 4}
USEFULNESS_FWD = ["QID47_1", "QID47_4", "QID47_5", "QID47_6", "QID47_9"]
USEFULNESS_REV = ["QID47_7", "QID47_8"]

outcomes = pd.DataFrame({
    "Chatbot showed an own stance (QID45_5)": X["QID45_5"],
    "Chatbot advised a party (QID45_6)": X["QID45_6"],
    "Perceived steering (QID35, 4 items)": X[["QID35_1", "QID35_2", "QID35_3", "QID35_4"]].mean(axis=1),
    "Perceived factuality (QID45_1, QID45_4)": X[["QID45_1", "QID45_4"]].mean(axis=1),
    "Ease of use (QID46, 4 items)": X[["QID46_1", "QID46_4", "QID46_5", "QID46_6"]].mean(axis=1),
    "Usefulness (QID47, 7 items)": pd.concat([X[USEFULNESS_FWD], reverse(X[USEFULNESS_REV], -3, 3)], axis=1).mean(axis=1),
    "Attitude toward use (QID48_1, QID48_4)": X[["QID48_1", "QID48_4"]].mean(axis=1),
    "Behavioral intention (QID48_5-7)": X[["QID48_5", "QID48_6", "QID48_7"]].mean(axis=1),
    "Change in political interest (QID41 - QID21)": X["QID41"] - X["QID21"],
    "Change in vote certainty (QID43 - QID27)": (complete["QID43"].map(CERTAINTY) - complete["QID27"].map(CERTAINTY)).astype(float),
})

# the two behavioural measures, joined the same way as in dialogue_strategies.ipynb
interaction = pd.read_csv(FINAL_DIR / "user_interaction.csv")
turns = interaction[interaction["role"] == "user"].copy()
turns["words"] = turns["content"].astype(str).str.split().str.len()
words = turns.groupby("username")["words"].median()
outcomes["Words per user turn (participant median)"] = complete["username"].map(words).to_numpy()

user_tbl, question_tbl = read(DB_DIR / "user.csv"), read(DB_DIR / "question.csv")
survey_cfg, raw_votes = read(DB_DIR / "survey.csv"), read(DB_DIR / "user_vote.csv")
annotations = read(FINAL_DIR / "annotations_export.csv")
votes = raw_votes.assign(
    username=lambda d: d["createdById"].map(user_tbl.set_index("id")["username"]),
    condition=lambda d: d["surveyId"].map(survey_cfg.set_index("id")["name"]),
    questionIndex=lambda d: d["questionId"].map(question_tbl.set_index("id")["index"]),
    vote5=lambda d: pd.to_numeric(d["value"]) * 4 + 1)
chat_votes = votes[votes["questionIndex"].isin(annotations["questionIndex"].unique())
                   & votes["condition"].isin(ARMS) & votes["username"].isin(complete["username"])]
neutral_share = chat_votes.groupby("username")["vote5"].apply(lambda s: (s == 3).mean())
outcomes["Neutral VAA answers (share of 4)"] = complete["username"].map(neutral_share).to_numpy()
assert outcomes.notna().all().all()

print("covariate means by arm (the imbalance the paper reports)")
display(covariates.groupby(cond.values).mean().T.round(3))

covariate means by arm (the imbalance the paper reports)


,Assistant,Interview
female,0.261,0.462
university,0.130,0.269
left-right (0-9),3.913,4.731
"CAVAA attitude, pre (-2..2)",0.467,0.183


## 2. Estimator


In [3]:
def pooled_sd(y, arm):
    a, b = y[arm == 1], y[arm == 0]
    na, nb = len(a), len(b)
    return np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))


def hedges_correction(n):
    return 1 - 3 / (4 * n - 9)


def hedges_g(y, arm):
    sd = pooled_sd(y, arm)
    if not sd > 0:
        return np.nan
    return (y[arm == 1].mean() - y[arm == 0].mean()) / sd * hedges_correction(len(y))


def ols(y, design):
    beta, *_ = np.linalg.lstsq(design, y, rcond=None)
    return beta


def adjusted_g(y, arm, Z):
    """Condition coefficient of y ~ 1 + arm + Z, divided by the pooled within-arm SD of y."""
    sd = pooled_sd(y, arm)
    if not sd > 0:
        return np.nan
    design = np.column_stack([np.ones(len(y)), arm, Z])
    return ols(y, design)[1] / sd * hedges_correction(len(y))


def compare(y, arm, Z, label):
    """One row: unadjusted g and adjusted g_adj, each with a bootstrap CI over the same draws."""
    rng = np.random.default_rng(SEED + zlib.crc32(label.encode()))
    y, Z = np.asarray(y, dtype=float), np.asarray(Z, dtype=float)
    ia, ib = np.flatnonzero(arm == 1), np.flatnonzero(arm == 0)
    raw, adj = [], []
    for _ in range(N_BOOT):
        idx = np.concatenate([rng.choice(ia, len(ia), replace=True), rng.choice(ib, len(ib), replace=True)])
        raw.append(hedges_g(y[idx], arm[idx]))
        adj.append(adjusted_g(y[idx], arm[idx], Z[idx]))
    raw, adj = np.array(raw), np.array(adj)
    raw, adj = raw[np.isfinite(raw)], adj[np.isfinite(adj)]
    g, g_adj = hedges_g(y, arm), adjusted_g(y, arm, Z)
    return {"measure": label,
            "g": g, "ci_low": np.percentile(raw, 2.5), "ci_high": np.percentile(raw, 97.5),
            "g_adj": g_adj, "ci_low_adj": np.percentile(adj, 2.5), "ci_high_adj": np.percentile(adj, 97.5),
            "shift": g_adj - g,
            "sign_agree_adj": float(np.mean(np.sign(adj) == np.sign(g_adj)))}


def effect_table(frame, arm, Z):
    return pd.DataFrame([compare(frame[c], arm, Z, c) for c in frame.columns]).set_index("measure")

## 3. Unadjusted and adjusted effects


In [4]:
table = effect_table(outcomes, reactive, covariates)
display(table.round(3))

print(f"largest |shift| over the {len(table)} outcomes: {table['shift'].abs().max():.3f} "
      f"({table['shift'].abs().idxmax()})")
print(f"median |shift|: {table['shift'].abs().median():.3f}")
print("intervals clear of zero, unadjusted:",
      ", ".join(m for m, r in table.iterrows() if r["ci_low"] > 0 or r["ci_high"] < 0) or "none")
print("intervals clear of zero, adjusted:  ",
      ", ".join(m for m, r in table.iterrows() if r["ci_low_adj"] > 0 or r["ci_high_adj"] < 0) or "none")

,g,ci_low,ci_high,g_adj,ci_low_adj,ci_high_adj,shift,sign_agree_adj
measure,,,,,,,,
Chatbot showed an own stance (QID45_5),-0.530,-1.167,0.016,-0.412,-1.102,0.131,0.118,0.929
Chatbot advised a party (QID45_6),-0.187,-0.761,0.380,-0.145,-0.758,0.414,0.043,0.706
"Perceived steering (QID35, 4 items)",-0.400,-0.966,0.134,-0.239,-0.799,0.288,0.161,0.820
"Perceived factuality (QID45_1, QID45_4)",0.299,-0.267,0.894,0.198,-0.398,0.810,-0.101,0.748
"Ease of use (QID46, 4 items)",0.250,-0.299,0.857,0.245,-0.385,0.917,-0.004,0.763
"Usefulness (QID47, 7 items)",0.076,-0.483,0.683,-0.146,-0.630,0.368,-0.222,0.730
"Attitude toward use (QID48_1, QID48_4)",0.098,-0.467,0.675,-0.040,-0.518,0.437,-0.138,0.577
Behavioral intention (QID48_5-7),-0.026,-0.622,0.532,-0.232,-0.769,0.238,-0.206,0.841
Change in political interest (QID41 - QID21),-0.446,-1.143,0.123,-0.514,-1.266,0.122,-0.069,0.945


largest |shift| over the 12 outcomes: 0.261 (Neutral VAA answers (share of 4))
median |shift|: 0.110
intervals clear of zero, unadjusted: Words per user turn (participant median), Neutral VAA answers (share of 4)
intervals clear of zero, adjusted:   Words per user turn (participant median)


## 4. Contribution of each covariate


In [5]:
single = pd.DataFrame({
    cov: [adjusted_g(outcomes[m].to_numpy(dtype=float), reactive, covariates[[cov]].to_numpy()) - table.loc[m, "g"]
          for m in outcomes.columns]
    for cov in covariates.columns
}, index=outcomes.columns)
single["all four"] = table["shift"]
display(single.round(3))

,female,university,left-right (0-9),"CAVAA attitude, pre (-2..2)",all four
Chatbot showed an own stance (QID45_5),-0.076,0.068,0.103,-0.012,0.118
Chatbot advised a party (QID45_6),-0.085,0.072,0.041,0.008,0.043
"Perceived steering (QID35, 4 items)",-0.069,0.061,0.124,0.009,0.161
"Perceived factuality (QID45_1, QID45_4)",0.072,-0.023,-0.113,-0.025,-0.101
"Ease of use (QID46, 4 items)",0.058,-0.015,-0.023,-0.046,-0.004
"Usefulness (QID47, 7 items)",0.042,-0.044,-0.061,-0.174,-0.222
"Attitude toward use (QID48_1, QID48_4)",0.033,-0.009,-0.031,-0.165,-0.138
Behavioral intention (QID48_5-7),0.004,-0.040,-0.019,-0.165,-0.206
Change in political interest (QID41 - QID21),0.006,-0.031,-0.004,-0.036,-0.069
Change in vote certainty (QID43 - QID27),0.013,0.003,0.019,-0.026,0.027


## 5. Covariates as predictors of the outcomes


In [6]:
def residual_on_condition(v, arm):
    design = np.column_stack([np.ones(len(v)), arm])
    return v - design @ ols(v, design)


resid_out = outcomes.apply(lambda s: residual_on_condition(s.to_numpy(dtype=float), reactive))
resid_cov = covariates.apply(lambda s: residual_on_condition(s.to_numpy(dtype=float), reactive))
corr = pd.DataFrame({c: [np.corrcoef(resid_out[m], resid_cov[c])[0, 1] for m in outcomes.columns]
                     for c in covariates.columns}, index=outcomes.columns)
display(corr.round(2))
print(f"|r| >= 0.3 cells: {int((corr.abs() >= 0.3).sum().sum())} of {corr.size}")

,female,university,left-right (0-9),"CAVAA attitude, pre (-2..2)"
Chatbot showed an own stance (QID45_5),-0.19,0.20,0.32,0.05
Chatbot advised a party (QID45_6),-0.21,0.21,0.13,-0.03
"Perceived steering (QID35, 4 items)",-0.17,0.18,0.38,-0.03
"Perceived factuality (QID45_1, QID45_4)",0.17,-0.07,-0.34,0.09
"Ease of use (QID46, 4 items)",0.14,-0.04,-0.07,0.17
"Usefulness (QID47, 7 items)",0.10,-0.13,-0.19,0.65
"Attitude toward use (QID48_1, QID48_4)",0.08,-0.03,-0.09,0.61
Behavioral intention (QID48_5-7),0.01,-0.12,-0.06,0.61
Change in political interest (QID41 - QID21),0.02,-0.09,-0.01,0.13
Change in vote certainty (QID43 - QID27),0.03,0.01,0.06,0.10


|r| >= 0.3 cells: 6 of 48


## 6. Summary


In [7]:
own = "Chatbot showed an own stance (QID45_5)"
steer = "Perceived steering (QID35, 4 items)"
fact = "Perceived factuality (QID45_1, QID45_4)"
words_m = "Words per user turn (participant median)"
neutral_m = "Neutral VAA answers (share of 4)"

claims = {
    "the largest shift over all outcomes is below 0.3 SD": table["shift"].abs().max() < 0.3,
    "own stance keeps its sign and stays below -0.4 after adjustment": table.loc[own, "g_adj"] < -0.4,
    "perceived steering keeps its sign but shrinks to between -0.3 and -0.2": -0.3 < table.loc[steer, "g_adj"] < -0.2,
    "perceived factuality keeps its sign and stays above +0.15": table.loc[fact, "g_adj"] > 0.15,
    "words per turn keeps an interval clear of zero": table.loc[words_m, "ci_low_adj"] > 0,
    "the neutral-vote interval no longer excludes zero after adjustment":
        table.loc[neutral_m, "ci_high_adj"] > 0 and table.loc[neutral_m, "ci_high"] < 0,
    "no perception interval becomes clear of zero through adjustment":
        not any((r["ci_low_adj"] > 0 or r["ci_high_adj"] < 0) and not (r["ci_low"] > 0 or r["ci_high"] < 0)
                for _, r in table.iterrows()),
    "the pre-study chatbot attitude correlates at r >= 0.6 with all three multi-item TAM dimensions":
        (corr.loc[["Usefulness (QID47, 7 items)", "Attitude toward use (QID48_1, QID48_4)",
                   "Behavioral intention (QID48_5-7)"], "CAVAA attitude, pre (-2..2)"] >= 0.6).all(),
}
for claim, holds in claims.items():
    print(f"  {'ok ' if holds else 'STALE'}  {claim}")

print("\nfor the paper:")
print(f"  own stance            g = {table.loc[own, 'g']:+.2f} -> g_adj = {table.loc[own, 'g_adj']:+.2f} "
      f"[{table.loc[own, 'ci_low_adj']:+.2f}, {table.loc[own, 'ci_high_adj']:+.2f}]")
print(f"  perceived steering    g = {table.loc[steer, 'g']:+.2f} -> g_adj = {table.loc[steer, 'g_adj']:+.2f} "
      f"[{table.loc[steer, 'ci_low_adj']:+.2f}, {table.loc[steer, 'ci_high_adj']:+.2f}]")
print(f"  perceived factuality  g = {table.loc[fact, 'g']:+.2f} -> g_adj = {table.loc[fact, 'g_adj']:+.2f} "
      f"[{table.loc[fact, 'ci_low_adj']:+.2f}, {table.loc[fact, 'ci_high_adj']:+.2f}]")
print(f"  largest shift         {table['shift'].abs().max():.2f} SD ({table['shift'].abs().idxmax()})")

  ok   the largest shift over all outcomes is below 0.3 SD
  ok   own stance keeps its sign and stays below -0.4 after adjustment
  ok   perceived steering keeps its sign but shrinks to between -0.3 and -0.2
  ok   perceived factuality keeps its sign and stays above +0.15
  ok   words per turn keeps an interval clear of zero
  ok   the neutral-vote interval no longer excludes zero after adjustment
  ok   no perception interval becomes clear of zero through adjustment
  ok   the pre-study chatbot attitude correlates at r >= 0.6 with all three multi-item TAM dimensions

for the paper:
  own stance            g = -0.53 -> g_adj = -0.41 [-1.10, +0.13]
  perceived steering    g = -0.40 -> g_adj = -0.24 [-0.80, +0.29]
  perceived factuality  g = +0.30 -> g_adj = +0.20 [-0.40, +0.81]
  largest shift         0.26 SD (Neutral VAA answers (share of 4))
